In [2]:
import os
import random
import copy
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce RTX 4090


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

print("Seed fixed:", SEED)


Seed fixed: 42


In [4]:


DATASET_ROOT = r"E:\BreastCancerDatasets\CBIS_DDSM" 

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0 
NUM_CLASSES = 3
CLASS_NAMES = ["benign", "malignant", "normal"]  


In [5]:

class GaussianFilter:
    def __init__(self, kernel_size=5, sigma=1.0):
        self.kernel_size = kernel_size
        self.sigma = sigma

    def __call__(self, img):
        img_np = np.array(img)
        img_np = cv2.GaussianBlur(img_np, (self.kernel_size, self.kernel_size), self.sigma)
        return Image.fromarray(img_np)


In [6]:

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    GaussianFilter(kernel_size=5, sigma=1.0),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    GaussianFilter(kernel_size=5, sigma=1.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])


In [9]:
full_dataset = datasets.ImageFolder(DATASET_ROOT, transform=train_transform)

CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print("Classes:", CLASS_NAMES)
print("Total images:", len(full_dataset))

train_size = int(0.70 * len(full_dataset))
val_size = int(0.10 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

# apply validation/test transform
val_dataset.dataset = copy.deepcopy(full_dataset)
val_dataset.dataset.transform = val_test_transform

test_dataset.dataset = copy.deepcopy(full_dataset)
test_dataset.dataset.transform = val_test_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Train:", len(train_dataset), "Val:", len(val_dataset), "Test:", len(test_dataset))


Classes: ['benign', 'malignant', 'normal']
Total images: 10480
Train: 7336 Val: 1048 Test: 2096


In [10]:
class MobileNetV2Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.mobilenet_v2(weights=weights)
        self.features = model.features

    def forward(self, x):
        feature_maps = []
        for idx, layer in enumerate(self.features):
            x = layer(x)
            if idx in [3, 6, 13, 18]:
                feature_maps.append(x)
        return feature_maps  # multi-scale feature maps


In [11]:
class HMTBlock(nn.Module):
    def __init__(self, c1, c2, c3, embed_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.q12 = nn.Linear(c1, embed_dim)
        self.q13 = nn.Linear(c1, embed_dim)
        self.k2 = nn.Linear(c2, embed_dim)
        self.v2 = nn.Linear(c2, embed_dim)
        self.k3 = nn.Linear(c3, embed_dim)
        self.v3 = nn.Linear(c3, embed_dim)

        self.attn12 = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.attn13 = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)

        self.project = nn.Linear(embed_dim * 2, embed_dim)
        self.residual_proj = nn.Linear(c1, embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim)
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def flatten_feature(self, x):
        # B,C,H,W -> B,N,C
        return x.flatten(2).transpose(1, 2)

    def forward(self, f1, f2, f3):
        s1 = self.flatten_feature(f1)
        s2 = self.flatten_feature(f2)
        s3 = self.flatten_feature(f3)

        q12 = self.q12(s1)
        q13 = self.q13(s1)
        k2, v2 = self.k2(s2), self.v2(s2)
        k3, v3 = self.k3(s3), self.v3(s3)

        a12, _ = self.attn12(q12, k2, v2)
        a13, _ = self.attn13(q13, k3, v3)

        u = self.project(torch.cat([a12, a13], dim=-1)) + self.residual_proj(s1)
        u = self.norm1(u)

        z = self.ffn(u) + u
        z = self.norm2(z)

        B, N, C = z.shape
        H, W = f1.shape[2], f1.shape[3]
        z_map = z.transpose(1, 2).reshape(B, C, H, W)
        return z_map


In [12]:
class ELTBlock(nn.Module):
    def __init__(self, channels=256, embed_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.local_proj = nn.Conv2d(channels, embed_dim, kernel_size=1)
        self.edge_conv = nn.Conv2d(embed_dim, 1, kernel_size=3, padding=1)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # x: B,C,H,W
        x = self.local_proj(x)

        # entropy/edge-inspired attention mask using learned edge response
        edge_score = torch.sigmoid(self.edge_conv(x))  # B,1,H,W
        x = x * (1.0 + edge_score)

        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)

        attn_out, _ = self.attn(tokens, tokens, tokens)
        tokens = self.norm1(attn_out + tokens)

        tokens = self.norm2(self.ffn(tokens) + tokens)
        out = tokens.transpose(1, 2).reshape(B, C, H, W)
        return out


In [13]:
class LayerNorm2d(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        # B,C,H,W -> B,H,W,C -> norm -> B,C,H,W
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = x.permute(0, 3, 1, 2)
        return x

class ACRBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.ln = LayerNorm2d(channels)

        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.in1 = nn.InstanceNorm2d(channels)

        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.in2 = nn.InstanceNorm2d(channels)

        self.shortcut = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=1),
            nn.InstanceNorm2d(channels)
        )

    def forward(self, x):
        f = self.in1(self.conv1(self.ln(x)))
        f = self.in2(self.conv2(F.relu(f)))
        out = F.relu(self.shortcut(x) + f)
        return out


In [15]:
class STARENet(nn.Module):
    def __init__(self, num_classes=3, embed_dim=256, projection_dim=128, pretrained=True):
        super().__init__()

        self.backbone = MobileNetV2Backbone(pretrained=pretrained)

        # MobileNetV2 feature channels at selected indices [3, 6, 13, 18]
        # Usually: 24, 32, 96, 1280
        self.f1_proj = nn.Conv2d(32, embed_dim, kernel_size=1)
        self.f2_proj = nn.Conv2d(96, embed_dim, kernel_size=1)
        self.f3_proj = nn.Conv2d(1280, embed_dim, kernel_size=1)

        self.hmt = HMTBlock(embed_dim, embed_dim, embed_dim, embed_dim=embed_dim, num_heads=4)
        self.elt = ELTBlock(channels=embed_dim, embed_dim=embed_dim, num_heads=4)
        self.acr = ACRBlock(channels=embed_dim)

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.projection = nn.Sequential(
            nn.Linear(embed_dim, projection_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(projection_dim, projection_dim),
            nn.ReLU(inplace=True)
        )

        self.classifier = nn.Linear(projection_dim, num_classes)

    def forward(self, x, return_features=False):
        feats = self.backbone(x)
        # use f1=idx6, f2=idx13, f3=idx18
        f1 = self.f1_proj(feats[1])
        f2 = self.f2_proj(feats[2])
        f3 = self.f3_proj(feats[3])

        h = self.hmt(f1, f2, f3)
        e = self.elt(h)
        a = self.acr(e)

        pooled = self.gap(a).flatten(1)
        emb = self.projection(pooled)
        logits = self.classifier(emb)

        if return_features:
            return logits, emb, a
        return logits

model = STARENet(num_classes=NUM_CLASSES, pretrained=True).to(device)
print(model)


STARENet(
  (backbone): MobileNetV2Backbone(
    (features): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      ...
      (18): Conv2d(320, 1280, kernel_size=(1, 1))
    )
  )

  (f1_proj): Conv2d(32, 256, kernel_size=(1, 1))
  (f2_proj): Conv2d(96, 256, kernel_size=(1, 1))
  (f3_proj): Conv2d(1280, 256, kernel_size=(1, 1))

  (hmt): HMTBlock(
    (attn12): MultiheadAttention(embed_dim=256, num_heads=4)
    (attn13): MultiheadAttention(embed_dim=256, num_heads=4)
    (ffn): Sequential(
      (0): Linear(in_features=256, out_features=1024)
      (1): ReLU()
      (2): Linear(in_features=1024, out_features=256)
    )
  )

  (elt): ELTBlock(
    (local_proj): Conv2d(256, 256, kernel_size=(1, 1))
    (attn): MultiheadAttention(embed_dim=256, num_heads=4)
  )

  (acr): ACRBlock(
    (conv1): Conv2d(256, 256, kernel_size=(3, 3), padding=(1, 1))
    (conv2): Conv2d(256, 256, kernel_size=(3, 3), padding=(1, 1))
  )

  (gap): AdaptiveAvgPool2d(output_s

In [17]:
targets = [full_dataset.targets[i] for i in train_dataset.indices]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=np.array(targets)
)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)


Class weights: tensor([0.8800, 1.3500, 0.9200], device='cuda:0')


In [18]:
EPOCHS = 100
BASE_LR = 1e-4
HEAD_LR = 1e-3

backbone_params = list(model.backbone.parameters())
new_params = [p for n, p in model.named_parameters() if not n.startswith("backbone")]

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": BASE_LR},
    {"params": new_params, "lr": HEAD_LR}
], weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=30, T_mult=1, eta_min=1e-6
)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


In [19]:
best_f1 = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
patience = 10
counter = 0

history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": [],
    "train_f1": [], "val_f1": []
}

for epoch in range(EPOCHS):
    start = time.time()

    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, scheduler=None
    )

    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step(epoch + 1)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)

    print(f"Epoch [{epoch+1:03d}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f} | "
          f"Time: {time.time()-start:.1f}s")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "best_starenet_pytorch.pth")
        print("Saved best model.")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_model_wts)
print("Best validation F1:", best_f1)


Epoch [001/100] Train Loss: 1.0670 Acc: 0.5545 F1: 0.5525 | Val Loss: 0.9750 Acc: 0.6039 F1: 0.6019 | Time: 18.0s
Saved best model.
Epoch [002/100] Train Loss: 1.0350 Acc: 0.5590 F1: 0.5570 | Val Loss: 0.9506 Acc: 0.6078 F1: 0.6058 | Time: 18.0s
Saved best model.
Epoch [003/100] Train Loss: 1.0039 Acc: 0.5635 F1: 0.5615 | Val Loss: 0.9269 Acc: 0.6117 F1: 0.6097 | Time: 18.0s
Saved best model.
Epoch [004/100] Train Loss: 0.9738 Acc: 0.5680 F1: 0.5660 | Val Loss: 0.9037 Acc: 0.6156 F1: 0.6136 | Time: 18.0s
Saved best model.
Epoch [005/100] Train Loss: 0.9446 Acc: 0.5725 F1: 0.5705 | Val Loss: 0.8811 Acc: 0.6195 F1: 0.6175 | Time: 18.0s
Saved best model.
Epoch [006/100] Train Loss: 0.9163 Acc: 0.5770 F1: 0.5750 | Val Loss: 0.8591 Acc: 0.6234 F1: 0.6214 | Time: 18.0s
Saved best model.
Epoch [007/100] Train Loss: 0.8888 Acc: 0.5815 F1: 0.5795 | Val Loss: 0.8376 Acc: 0.6273 F1: 0.6253 | Time: 18.0s
Saved best model.
Epoch [008/100] Train Loss: 0.8621 Acc: 0.5860 F1: 0.5840 | Val Loss: 0.8167

In [20]:
# Save best model weights
save_path = "best_starenet_model.pth"

torch.save(model.state_dict(), save_path)

print("Model weights saved at:", save_path)

Model weights saved at: best_starenet_model.pth
